In [ ]:
import networkx as nx
import os

def merge_gexf_files(file_paths, output_path):
    """
    Merges multiple GEXF files into one.
    Nodes with the same ID will have their attributes merged.
    """
    merged_graph = nx.Graph()

    for path in file_paths:
        if not os.path.exists(path):
            print(f"Warning: File not found: {path}")
            continue

        print(f"Processing: {path}")
        # Load the graph
        current_graph = nx.read_gexf(path)

        # Merge nodes and attributes
        for node, data in current_graph.nodes(data=True):
            if merged_graph.has_node(node):
                # Update existing node with new attributes
                merged_graph.nodes[node].update(data)
            else:
                merged_graph.add_node(node, **data)

        # Merge edges
        for u, v, data in current_graph.edges(data=True):
            if not merged_graph.has_edge(u, v):
                merged_graph.add_edge(u, v, **data)

    # Save the result (creating output directory if needed)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    nx.write_gexf(merged_graph, output_path)
    print(f"Successfully created merged graph at: {output_path}")
    return merged_graph

# ── REGISTRO DE DIMENSIONES ─────────────────────────────────────────────
# Para añadir una nueva dimensión (postura, vestimenta...):
#   1. Crea pipeline/04_DIMENSION.ipynb que genere output/DIMENSION/grafo_DIMENSION.gexf
#   2. Añade la ruta aquí. El paso 3 fusiona todo automáticamente.
# ─────────────────────────────────────────────────────────────────────────
files_to_merge = [
    '/content/drive/MyDrive/TFM-Sara/output/transcripciones/knowledge_graph_palabras.gexf',   # dim: transcripcion
    '/content/drive/MyDrive/TFM-Sara/output/predicciones_año/grafo_años.gexf',                # dim: año
    # '/content/drive/MyDrive/TFM-Sara/output/postura/grafo_postura.gexf',                    # dim: postura  (futuro)
    # '/content/drive/MyDrive/TFM-Sara/output/vestimenta/grafo_vestimenta.gexf',              # dim: vestimenta (futuro)
]

output_file = '/content/drive/MyDrive/TFM-Sara/output/grafo_final/grafo_multidimensional.gexf'

# Execute merge
full_graph = merge_gexf_files(files_to_merge, output_file)

# ── Post-merge: normalizar nodos imagen ──────────────────────────────────
# Tras la fusión los nodos imagen pueden tener colores/atributos inconsistentes
# (azul claro del paso 1 vs rojo del paso 2). Los unificamos aquí.
from collections import Counter
dimension_counts = Counter()

for node, data in full_graph.nodes(data=True):
    dim = data.get('dimension', 'desconocida')
    dimension_counts[dim] += 1
    # Normalizar nodos imagen: color y group consistentes
    if data.get('group') == 1 or data.get('color') in ('lightblue', '#FF0000', '#4A90D9'):
        full_graph.nodes[node]['color']     = '#4A90D9'
        full_graph.nodes[node]['group']     = 1
        full_graph.nodes[node]['dimension'] = 'imagen'
        full_graph.nodes[node]['size']      = data.get('size', 25)

# Re-guardar con nodos normalizados
nx.write_gexf(full_graph, output_file)

print(f"Nodos por dimensión:")
for dim, count in sorted(dimension_counts.items()):
    print(f"  {dim:20s}: {count}")
from collections import Counter as _Counter2
rel_counts = _Counter2(d.get("relation", "sin_tipo") for _, _, d in full_graph.edges(data=True))
print(f"Total nodos  : {full_graph.number_of_nodes()}")
print(f"Total aristas: {full_graph.number_of_edges()}")
print("Aristas por tipo:")
for rel, cnt in sorted(rel_counts.items()):
    print(f"  {rel:25s}: {cnt}")

Processing: /content/drive/MyDrive/TFM-Sara/output/transcripciones/knowledge_graph_palabras.gexf
Processing: /content/drive/MyDrive/TFM-Sara/output/predicciones_año/grafo_años.gexf
Successfully created merged graph at: /content/drive/MyDrive/TFM-Sara/output/grafo_final/grafo_multidimensional.gexf
Total nodes: 1278
Total edges: 1404


In [ ]:
# ── Inyectar atributos viz: en el GEXF ─────────────────────────────────────
# El namespace viz: (GEXF 1.2) embebe color, tamaño y forma directamente
# en el fichero. Gephi y Gephi Lite los aplican al abrir sin configuración.
import xml.etree.ElementTree as ET

GEXF_NS  = 'http://www.gexf.net/1.2draft'
VIZ_NS   = 'http://www.gexf.net/1.2draft/viz'
ET.register_namespace('',    GEXF_NS)
ET.register_namespace('viz', VIZ_NS)

# Paleta de colores por dimensión (r, g, b) y forma
DIM_STYLE = {
    'imagen':       {'r': 74,  'g': 144, 'b': 217, 'shape': 'disc'},
    'transcripcion':{'r': 92,  'g': 184, 'b': 92,  'shape': 'disc'},
    'año':          {'r': 26,  'g': 58,  'b': 92,  'shape': 'square'},
    'postura':      {'r': 230, 'g': 126, 'b': 34,  'shape': 'diamond'},
    'vestimenta':   {'r': 155, 'g': 89,  'b': 182, 'shape': 'triangle'},
}

def _hex_to_rgb(hex_color):
    h = hex_color.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def inject_viz_namespace(gexf_path):
    tree = ET.parse(gexf_path)
    root = tree.getroot()

    # Añadir declaración del namespace viz al elemento raíz
    root.set('xmlns:viz', VIZ_NS)

    graph_el = root.find(f'{{{GEXF_NS}}}graph')
    nodes_el  = graph_el.find(f'{{{GEXF_NS}}}nodes')

    for node_el in nodes_el.findall(f'{{{GEXF_NS}}}node'):
        node_id = node_el.get('id')
        if node_id not in full_graph.nodes:
            continue

        data  = full_graph.nodes[node_id]
        dim   = data.get('dimension', 'imagen')
        style = DIM_STYLE.get(dim, DIM_STYLE['imagen'])
        size  = float(data.get('size', 25))

        # viz:color
        color_el = ET.SubElement(node_el, f'{{{VIZ_NS}}}color')
        color_el.set('r', str(style['r']))
        color_el.set('g', str(style['g']))
        color_el.set('b', str(style['b']))
        color_el.set('a', '255')

        # viz:size
        size_el = ET.SubElement(node_el, f'{{{VIZ_NS}}}size')
        size_el.set('value', f'{size:.1f}')

        # viz:shape
        shape_el = ET.SubElement(node_el, f'{{{VIZ_NS}}}shape')
        shape_el.set('value', style['shape'])

    tree.write(gexf_path, encoding='utf-8', xml_declaration=True)
    print(f'✅ viz: namespace inyectado en {os.path.basename(gexf_path)}')
    print('   Gephi / Gephi Lite abrirán el grafo ya con colores y formas aplicados.')

inject_viz_namespace(output_file)